# ============================================
# AGRI-WATCH Sénégal
# Notebook 06 - Calcul des indicateurs
# Auteure : Adji Fatou NGOM - ANSD
# ============================================

In [6]:
# ============================================
# Indicateurs calculés :
# 1. SPI  — Standardized Precipitation Index
#    Source : McKee et al. (1993)
#             OMM WMO-No.1090 (2012)
#
# 2. Anomalie NDVI
#    Source : FAO/GIEWS ASIS (2017)
#
# 3. Bilan hydrique
#    Source : Allen et al. (1998) FAO-56
#
# 4. ISSA — Indice de Stress Agricole
#    Sénégalais (indicateur composite)
#    Source : Svoboda et al. (2002)
#             Kogan (1995)
#
# 5. Cartes de risque et alertes précoces
# ============================================

import sys
import warnings
warnings.filterwarnings('ignore')

if "C:/AGRI-WATCH" not in sys.path:
    sys.path.append("C:/AGRI-WATCH")

for module in list(sys.modules.keys()):
    if module.startswith('src'):
        del sys.modules[module]

print("Path et cache initialises avec succes.")

Path et cache initialises avec succes.


In [7]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm
from pathlib import Path
from datetime import datetime
from scipy import stats
from scipy.stats import (
    gamma, kstest, pearsonr, spearmanr
)
from scipy.special import gammainc
import warnings
warnings.filterwarnings('ignore')

from src.config import (
    SHAPEFILE_DEPARTEMENTS,
    SHAPEFILE_REGIONS,
    COL_NOM_DEPARTEMENT,
    COL_NOM_REGION,
    CHIRPS_DIR,
    SENTINEL2_DIR,
    ERA5_DIR,
    OUTPUTS_DIR,
    COULEURS_RISQUE,
    ISSA_POIDS,
    ALERTE_SEUILS,
    creer_dossiers
)
from src.logger import get_logger

creer_dossiers()
logger = get_logger("calcul_indicateurs")
logger.info("Notebook 06 - Calcul indicateurs - Demarrage")
print("Imports termines avec succes.")

Structure des dossiers AGRI-WATCH verifiee avec succes.
Racine du projet : C:\AGRI-WATCH
[2026-05-08 22:51:43] [INFO] [agriwatch.calcul_indicateurs] Notebook 06 - Calcul indicateurs - Demarrage
Imports termines avec succes.


In [8]:
# ============================================
# Chargement des données
# ============================================

racine = Path("C:/AGRI-WATCH")

logger.info("Chargement donnees pour calcul indicateurs...")

# ── Données satellitaires ─────────────────────────────────────
chirps_ref = pd.read_csv(
    racine / "data/raw/chirps/chirps_reference_2000_2024.csv"
)
chirps_ana = pd.read_csv(
    racine / "data/raw/chirps/chirps_analyse_2022_2024.csv"
)
modis_ref = pd.read_csv(
    racine / "data/raw/sentinel2/modis_ndvi_reference_2000_2024.csv"
)
era5_ref = pd.read_csv(
    racine / "data/raw/era5/era5_reference_2000_2024.csv"
)
era5_ana = pd.read_csv(
    racine / "data/raw/era5/era5_analyse_2022_2024.csv"
)

# ── Données agricoles pour backtesting ───────────────────────
dataset_ml = pd.read_csv(
    racine / "data/processed/dataset_ml_final.csv",
    encoding = "utf-8-sig"
)

# ── Shapefiles ────────────────────────────────────────────────
departements = gpd.read_file(SHAPEFILE_DEPARTEMENTS)
regions      = gpd.read_file(SHAPEFILE_REGIONS)

# ── Vérifications ─────────────────────────────────────────────
print("=" * 60)
print("CHARGEMENT DES DONNEES")
print("=" * 60)
print(f"CHIRPS reference  : {len(chirps_ref):,} lignes")
print(f"CHIRPS analyse    : {len(chirps_ana):,} lignes")
print(f"MODIS reference   : {len(modis_ref):,} lignes")
print(f"ERA5 reference    : {len(era5_ref):,} lignes")
print(f"ERA5 analyse      : {len(era5_ana):,} lignes")
print(f"Dataset ML        : {len(dataset_ml):,} lignes")
print(f"Departements      : {len(departements)} polygones")
print(f"\nPeriode reference : "
      f"{chirps_ref['annee'].min()} - "
      f"{chirps_ref['annee'].max()}")
print(f"Periode analyse   : "
      f"{chirps_ana['annee'].min()} - "
      f"{chirps_ana['annee'].max()}")

# ── Paramètres globaux ────────────────────────────────────────
# Période de référence pour les normales
# On utilise 2000-2020 pour avoir les
# rendements DAPSA comme référence
# de validation (backtesting)
ANNEE_REF_DEBUT = 2000
ANNEE_REF_FIN   = 2020

# Saison agricole : juin à octobre
MOIS_SAISON = [6, 7, 8, 9, 10]

print(f"\nParametres :")
print(f"Periode reference : {ANNEE_REF_DEBUT}-{ANNEE_REF_FIN}")
print(f"Mois saison       : {MOIS_SAISON}")
print(f"Poids ISSA        : {ISSA_POIDS}")

logger.info("Donnees chargees avec succes.")

[2026-05-08 22:57:03] [INFO] [agriwatch.calcul_indicateurs] Chargement donnees pour calcul indicateurs...
CHARGEMENT DES DONNEES
CHIRPS reference  : 5,623 lignes
CHIRPS analyse    : 674 lignes
MODIS reference   : 5,615 lignes
ERA5 reference    : 5,625 lignes
ERA5 analyse      : 675 lignes
Dataset ML        : 1,457 lignes
Departements      : 45 polygones

Periode reference : 2000 - 2024
Periode analyse   : 2022 - 2024

Parametres :
Periode reference : 2000-2020
Mois saison       : [6, 7, 8, 9, 10]
Poids ISSA        : {'spi': 0.4, 'ndvi_anomalie': 0.35, 'bilan_hydrique': 0.25}
[2026-05-08 22:57:06] [INFO] [agriwatch.calcul_indicateurs] Donnees chargees avec succes.


In [9]:
# ============================================
# Calcul SPI
# Standardized Precipitation Index
# ============================================
# Le SPI mesure les déficits ou excédents
# pluviométriques par rapport à la normale
# historique pour chaque département.
#
# Méthode :
# 1. Ajustement d'une loi Gamma sur les
#    précipitations historiques 2000-2020
#    par département
# 2. Test KS pour valider l'ajustement
# 3. Transformation en scores Z (SPI)
#
# Sources :
#   McKee, T.B., Doesken, N.J. & Kleist, J.
#   (1993). The relationship of drought
#   frequency and duration to time scales.
#   8th Conference on Applied Climatology.
#   American Meteorological Society.
#
#   OMM (2012). Standardized Precipitation
#   Index User Guide. WMO-No.1090. Genève.
#   URL : public.wmo.int
#
#   Thom, H.C.S. (1958). A note on the
#   Gamma distribution. Monthly Weather
#   Review, 86(4), 117-122.
# ============================================

def calculer_spi(
    chirps: pd.DataFrame,
    annee_ref_debut: int,
    annee_ref_fin: int
) -> pd.DataFrame:
    """
    Calcule le SPI saisonnier pour chaque
    département et chaque année.

    Le SPI saisonnier utilise les précipitations
    totales de la saison juin-octobre comme
    variable d'entrée — ce qui est plus adapté
    aux cultures pluviales que le SPI mensuel.

    Étapes de calcul :
    1. Calcul précipitations totales par saison
    2. Ajustement loi Gamma sur 2000-2020
    3. Test KS pour valider l'ajustement
    4. Transformation en probabilité cumulative
    5. Transformation en score Z (SPI)

    Sources :
        McKee et al. (1993). AMS.
        OMM WMO-No.1090 (2012).
        Thom (1958). MWR 86(4).

    Paramètres :
        chirps          (DataFrame) : Données CHIRPS
        annee_ref_debut (int)       : Début période référence
        annee_ref_fin   (int)       : Fin période référence

    Retourne :
        DataFrame : SPI par département et année
    """
    logger.info("Calcul SPI saisonnier...")

    # ── Précipitations totales saisonnières ───────────────────
    pluie_saison = chirps.groupby(
        ["departement", "annee"]
    )["precipitation_mm"].sum().reset_index()
    pluie_saison.columns = [
        "departement", "annee", "pluie_totale"
    ]

    # ── Référence 2000-2020 ───────────────────────────────────
    pluie_ref = pluie_saison[
        (pluie_saison["annee"] >= annee_ref_debut) &
        (pluie_saison["annee"] <= annee_ref_fin)
    ]

    resultats_spi  = []
    stats_ajustement = []

    for dept in pluie_saison["departement"].unique():

        # Précipitations historiques du département
        pluie_dept_ref = pluie_ref[
            pluie_ref["departement"] == dept
        ]["pluie_totale"].values

        # Précipitations toutes années
        pluie_dept_all = pluie_saison[
            pluie_saison["departement"] == dept
        ].copy()

        # ── Gestion des zéros ─────────────────────────────────
        # La loi Gamma ne peut pas gérer les zéros
        # Source : OMM WMO-No.1090 (2012) Section 4.2
        # Méthode de correction : on ajoute un epsilon
        # pour les valeurs nulles avant ajustement
        nb_zeros = (pluie_dept_ref == 0).sum()
        if nb_zeros > 0:
            pluie_dept_ref_corr = pluie_dept_ref.copy()
            pluie_dept_ref_corr[pluie_dept_ref_corr == 0] = 0.001
        else:
            pluie_dept_ref_corr = pluie_dept_ref

        # ── Ajustement loi Gamma ──────────────────────────────
        # Source : Thom (1958). MWR 86(4)
        try:
            alpha, loc, beta = gamma.fit(
                pluie_dept_ref_corr,
                floc = 0  # loc fixé à 0 pour loi Gamma
            )

            # ── Test Kolmogorov-Smirnov ───────────────────────
            # Vérifie si la loi Gamma est bien ajustée
            # Source : Wilks (2011). Academic Press
            ks_stat, ks_p = kstest(
                pluie_dept_ref_corr,
                "gamma",
                args=(alpha, loc, beta)
            )
            ajustement_ok = ks_p > 0.05

            stats_ajustement.append({
                "departement"  : dept,
                "alpha"        : round(alpha, 3),
                "beta"         : round(beta, 3),
                "nb_zeros"     : nb_zeros,
                "ks_stat"      : round(ks_stat, 3),
                "ks_p"         : round(ks_p, 4),
                "ajustement_ok": ajustement_ok
            })

            # ── Calcul SPI pour toutes les années ─────────────
            for _, row in pluie_dept_all.iterrows():
                pluie = max(row["pluie_totale"], 0.001)

                # Probabilité cumulative loi Gamma
                prob = gamma.cdf(pluie, alpha, loc, beta)

                # Correction pour éviter prob = 0 ou 1
                # Source : OMM WMO-No.1090 (2012)
                prob = np.clip(prob, 0.0013499, 0.9986501)

                # Transformation en score Z (SPI)
                # Source : McKee et al. (1993)
                spi = stats.norm.ppf(prob)

                resultats_spi.append({
                    "departement" : dept,
                    "annee"       : row["annee"],
                    "pluie_totale": row["pluie_totale"],
                    "spi"         : round(spi, 3)
                })

        except Exception as e:
            logger.warning(
                f"Ajustement Gamma echoue pour {dept} : {e}"
            )
            continue

    df_spi   = pd.DataFrame(resultats_spi)
    df_stats = pd.DataFrame(stats_ajustement)

    return df_spi, df_stats


# ── Calcul SPI ────────────────────────────────────────────────
df_spi, df_stats_gamma = calculer_spi(
    chirps          = chirps_ref,
    annee_ref_debut = ANNEE_REF_DEBUT,
    annee_ref_fin   = ANNEE_REF_FIN
)

# ── Résultats ─────────────────────────────────────────────────
print("=" * 65)
print("CALCUL SPI — RESULTATS")
print("Sources : McKee et al. (1993) | OMM WMO-No.1090 (2012)")
print("          Thom (1958). MWR 86(4)")
print("=" * 65)

print(f"\n1. DATASET SPI")
print(f"   Lignes         : {len(df_spi):,}")
print(f"   Departements   : {df_spi['departement'].nunique()}")
print(f"   Periode        : {df_spi['annee'].min()} - "
      f"{df_spi['annee'].max()}")

print(f"\n2. STATISTIQUES SPI")
print(f"   Moyenne        : {df_spi['spi'].mean():.3f}")
print(f"   Ecart-type     : {df_spi['spi'].std():.3f}")
print(f"   Min            : {df_spi['spi'].min():.3f}")
print(f"   Max            : {df_spi['spi'].max():.3f}")

print(f"\n3. VALIDATION AJUSTEMENT GAMMA")
print(f"   Test KS — Source : Wilks (2011)")
nb_ok  = df_stats_gamma["ajustement_ok"].sum()
nb_tot = len(df_stats_gamma)
print(f"   Ajustements OK : {nb_ok}/{nb_tot} "
      f"({nb_ok/nb_tot*100:.1f}%)")
print(f"   Ajustements NON OK :")
non_ok = df_stats_gamma[~df_stats_gamma["ajustement_ok"]]
if len(non_ok) > 0:
    print(non_ok[
        ["departement", "ks_stat", "ks_p"]
    ].to_string(index=False))
else:
    print("   Aucun — tous les ajustements sont valides !")

print(f"\n4. CLASSIFICATION SPI")
print(f"   Source : OMM WMO-No.1090 (2012)")
classifications = [
    ("Secheresse extreme  (SPI < -2.0)",
     (df_spi["spi"] < -2.0).sum()),
    ("Secheresse severe   (-2.0 < SPI < -1.5)",
     ((df_spi["spi"] >= -2.0) & (df_spi["spi"] < -1.5)).sum()),
    ("Secheresse moderee  (-1.5 < SPI < -1.0)",
     ((df_spi["spi"] >= -1.5) & (df_spi["spi"] < -1.0)).sum()),
    ("Normal              (-1.0 < SPI < +1.0)",
     ((df_spi["spi"] >= -1.0) & (df_spi["spi"] < 1.0)).sum()),
    ("Excedent modere     (+1.0 < SPI < +1.5)",
     ((df_spi["spi"] >= 1.0) & (df_spi["spi"] < 1.5)).sum()),
    ("Excedent severe     (+1.5 < SPI < +2.0)",
     ((df_spi["spi"] >= 1.5) & (df_spi["spi"] < 2.0)).sum()),
    ("Excedent extreme    (SPI > +2.0)",
     (df_spi["spi"] > 2.0).sum()),
]
for label, nb in classifications:
    pct = nb / len(df_spi) * 100
    print(f"   {label} : {nb:>4} ({pct:.1f}%)")

print(f"\n5. ANNEES EXTREMES NATIONALES")
spi_national = df_spi.groupby("annee")["spi"].mean()
print(f"   Annee SPI min  : "
      f"{spi_national.idxmin()} "
      f"(SPI={spi_national.min():.3f})")
print(f"   Annee SPI max  : "
      f"{spi_national.idxmax()} "
      f"(SPI={spi_national.max():.3f})")

print("\nSources :")
print("  McKee et al. (1993). 8th Conf. Applied Climatology. AMS.")
print("  OMM (2012). WMO-No.1090. Geneve.")
print("  Thom (1958). Monthly Weather Review 86(4).")
print("  Wilks (2011). Statistical Methods. Academic Press.")
print("=" * 65)

# ── Sauvegarde ────────────────────────────────────────────────
output_spi = Path(
    "C:/AGRI-WATCH/data/processed/spi_departements_2000_2024.csv"
)
df_spi.to_csv(output_spi, index=False, encoding="utf-8-sig")
logger.info(
    f"SPI calcule et sauvegarde : "
    f"{len(df_spi)} lignes"
)

[2026-05-09 01:36:56] [INFO] [agriwatch.calcul_indicateurs] Calcul SPI saisonnier...
CALCUL SPI — RESULTATS
Sources : McKee et al. (1993) | OMM WMO-No.1090 (2012)
          Thom (1958). MWR 86(4)

1. DATASET SPI
   Lignes         : 1,125
   Departements   : 45
   Periode        : 2000 - 2024

2. STATISTIQUES SPI
   Moyenne        : 0.102
   Ecart-type     : 0.986
   Min            : -3.000
   Max            : 2.771

3. VALIDATION AJUSTEMENT GAMMA
   Test KS — Source : Wilks (2011)
   Ajustements OK : 45/45 (100.0%)
   Ajustements NON OK :
   Aucun — tous les ajustements sont valides !

4. CLASSIFICATION SPI
   Source : OMM WMO-No.1090 (2012)
   Secheresse extreme  (SPI < -2.0) :   16 (1.4%)
   Secheresse severe   (-2.0 < SPI < -1.5) :   47 (4.2%)
   Secheresse moderee  (-1.5 < SPI < -1.0) :   94 (8.4%)
   Normal              (-1.0 < SPI < +1.0) :  752 (66.8%)
   Excedent modere     (+1.0 < SPI < +1.5) :  126 (11.2%)
   Excedent severe     (+1.5 < SPI < +2.0) :   61 (5.4%)
   Excedent e

In [10]:
# Affichage complet
print("CLASSIFICATION SPI COMPLETE :")
classifications = [
    ("Secheresse extreme  (SPI < -2.0)",
     (df_spi["spi"] < -2.0).sum()),
    ("Secheresse severe   (-2.0 < SPI < -1.5)",
     ((df_spi["spi"] >= -2.0) &
      (df_spi["spi"] < -1.5)).sum()),
    ("Secheresse moderee  (-1.5 < SPI < -1.0)",
     ((df_spi["spi"] >= -1.5) &
      (df_spi["spi"] < -1.0)).sum()),
    ("Normal              (-1.0 < SPI < +1.0)",
     ((df_spi["spi"] >= -1.0) &
      (df_spi["spi"] < 1.0)).sum()),
    ("Excedent modere     (+1.0 < SPI < +1.5)",
     ((df_spi["spi"] >= 1.0) &
      (df_spi["spi"] < 1.5)).sum()),
    ("Excedent severe     (+1.5 < SPI < +2.0)",
     ((df_spi["spi"] >= 1.5) &
      (df_spi["spi"] < 2.0)).sum()),
    ("Excedent extreme    (SPI > +2.0)",
     (df_spi["spi"] > 2.0).sum()),
]
for label, nb in classifications:
    pct = nb / len(df_spi) * 100
    print(f"   {label} : {nb:>4} ({pct:.1f}%)")

print("\nANNEES EXTREMES NATIONALES :")
spi_national = df_spi.groupby("annee")["spi"].mean()
for annee, spi in spi_national.sort_values().items():
    print(f"   {annee} : SPI={spi:+.3f}")

CLASSIFICATION SPI COMPLETE :
   Secheresse extreme  (SPI < -2.0) :   16 (1.4%)
   Secheresse severe   (-2.0 < SPI < -1.5) :   47 (4.2%)
   Secheresse moderee  (-1.5 < SPI < -1.0) :   94 (8.4%)
   Normal              (-1.0 < SPI < +1.0) :  752 (66.8%)
   Excedent modere     (+1.0 < SPI < +1.5) :  126 (11.2%)
   Excedent severe     (+1.5 < SPI < +2.0) :   61 (5.4%)
   Excedent extreme    (SPI > +2.0) :   29 (2.6%)

ANNEES EXTREMES NATIONALES :
   2014 : SPI=-1.597
   2019 : SPI=-0.820
   2011 : SPI=-0.819
   2007 : SPI=-0.781
   2018 : SPI=-0.674
   2004 : SPI=-0.590
   2002 : SPI=-0.449
   2001 : SPI=-0.447
   2006 : SPI=-0.389
   2017 : SPI=-0.172
   2016 : SPI=-0.126
   2013 : SPI=-0.095
   2003 : SPI=+0.111
   2000 : SPI=+0.111
   2021 : SPI=+0.344
   2024 : SPI=+0.497
   2008 : SPI=+0.510
   2023 : SPI=+0.526
   2009 : SPI=+0.791
   2005 : SPI=+0.844
   2015 : SPI=+0.935
   2012 : SPI=+1.005
   2022 : SPI=+1.170
   2010 : SPI=+1.234
   2020 : SPI=+1.433


## Calcul SPI 

### Méthode

Le SPI (Standardized Precipitation Index)
est l'indicateur officiel de sécheresse
pluviométrique de l'Organisation
Météorologique Mondiale.

**Calcul :**
1. Ajustement loi Gamma sur précipitations
   historiques 2000-2020 par département
2. Test Kolmogorov-Smirnov pour valider
   l'ajustement
3. Transformation en score Z (SPI)

**Sources :**
McKee et al. (1993). 8th Conf. Applied
Climatology. AMS.
OMM (2012). WMO-No.1090. Genève.
Thom (1958). Monthly Weather Review 86(4).

### Validation statistique

| Indicateur | Valeur | Attendu | Statut |
|---|---|---|---|
| Ajustements Gamma | 45/45 (100%) | > 90% | Excellent |
| Moyenne SPI | 0.102 | ≈ 0 | Excellent |
| Écart-type SPI | 0.986 | ≈ 1 | Excellent |
| Valeurs normales | 66.8% | ≈ 68% | Excellent |

### Classification SPI — période 2000-2024

| Classe | Nb | % | Source seuils |
|---|---|---|---|
| Sécheresse extrême (< -2.0) | 16 | 1.4% | OMM WMO-No.1090 |
| Sécheresse sévère (-2.0 à -1.5) | 47 | 4.2% | OMM WMO-No.1090 |
| Sécheresse modérée (-1.5 à -1.0) | 94 | 8.4% | OMM WMO-No.1090 |
| Normal (-1.0 à +1.0) | 752 | 66.8% | OMM WMO-No.1090 |
| Excédent modéré (+1.0 à +1.5) | 126 | 11.2% | OMM WMO-No.1090 |
| Excédent sévère (+1.5 à +2.0) | 61 | 5.4% | OMM WMO-No.1090 |
| Excédent extrême (> +2.0) | 29 | 2.6% | OMM WMO-No.1090 |

### Validation croisée avec analyses précédentes

Les années extrêmes du SPI sont
parfaitement cohérentes avec les
résultats des Notebooks 03 et 05 —

**2014 — SPI le plus négatif (-1.597)**
Confirmé dans le Notebook 03 comme
l'année la plus sèche avec 516mm
soit -22.4% sous la normale historique.

**2020 — SPI le plus positif (+1.433)**
Confirmé dans le Notebook 03 comme
l'année la plus humide avec 818mm
soit +22.9% au-dessus de la normale,
et dans le Notebook 04 comme la
meilleure année pour les rendements
agricoles.

**2022 — SPI = +1.170**
Confirmé dans le Notebook 05 comme
excellente saison avec une anomalie
pluviométrique de +1.18σ.

Cette triple cohérence entre le SPI
et les analyses précédentes valide
la qualité du calcul et la fiabilité
des données CHIRPS utilisées.

### Limitation importante

La corrélation du SPI avec les
rendements agricoles est de rs=0.226
— significative mais faible. Cette
limitation vient du fait que le SPI
normalise par département et supprime
l'effet du gradient pluviométrique
nord-sud. C'est pourquoi l'anomalie
pluviométrique normalisée (rs=0.658)
sera utilisée dans le calcul de l'ISSA
tandis que le SPI sera présenté
séparément comme indicateur de
surveillance pluviométrique officiel
conforme aux standards OMM.

### Sources officielles

McKee, T.B., Doesken, N.J. & Kleist, J.
(1993). 8th Conf. Applied Climatology. AMS.

OMM (2012). Standardized Precipitation
Index User Guide. WMO-No.1090. Genève.

Thom, H.C.S. (1958). Monthly Weather
Review 86(4), 117-122.

Wilks (2011). Statistical Methods in the
Atmospheric Sciences. 3rd ed. Academic Press.

In [12]:
# ============================================
# Calcul Anomalie NDVI
# ============================================
# L'anomalie NDVI mesure l'écart entre
# le NDVI actuel et la normale historique
# du département pour le même mois.
#
# C'est le meilleur prédicteur des rendements
# agricoles selon nos corrélations Spearman
# du Notebook 05 (rs=0.707 arachide)
#
# Méthode :
#   anomalie_ndvi =
#   (NDVI_actuel - NDVI_moy_historique)
#   / NDVI_std_historique
#
# Source :
#   FAO/GIEWS Agricultural Stress Index
#   System (ASIS) (2017).
#   URL : fao.org/giews/earthobservation
#
#   Tucker, C.J. & Sellers, P.J. (1986).
#   Satellite remote sensing of primary
#   production. International Journal of
#   Remote Sensing, 7(11), 1395-1416.
#
#   Fensholt & Proud (2012). Evaluation of
#   Earth Observation based global long term
#   vegetation trends. Remote Sensing of
#   Environment 119, 131-147.
# ============================================

def calculer_anomalie_ndvi(
    modis: pd.DataFrame,
    annee_ref_debut: int,
    annee_ref_fin: int
) -> tuple:
    """
    Calcule l'anomalie NDVI normalisée
    pour chaque département, mois et année.

    L'anomalie NDVI est calculée comme le
    score Z du NDVI par rapport à la normale
    historique du département pour le même mois.

    Cette méthode est standard en télédétection
    agricole et utilisée par le système
    FAO/GIEWS ASIS pour la surveillance
    mondiale des cultures.

    Étapes de calcul :
    1. Calcul normale historique NDVI
       (moyenne + écart-type) par département
       et par mois sur 2000-2020
    2. Calcul anomalie normalisée = score Z
    3. Calcul anomalie saisonnière
       (moyenne juin-octobre)
    4. Validation statistique

    Sources :
        FAO/GIEWS ASIS (2017).
        Tucker & Sellers (1986). IJRS 7(11).
        Fensholt & Proud (2012). RSE 119.

    Paramètres :
        modis           (DataFrame) : MODIS NDVI
        annee_ref_debut (int)       : Début référence
        annee_ref_fin   (int)       : Fin référence

    Retourne :
        tuple : (anomalie mensuelle, anomalie saisonnière,
                 normales historiques)
    """
    logger.info("Calcul anomalie NDVI...")

    # ── Normales historiques 2000-2020 ────────────────────────
    # Moyenne et écart-type par département et par mois
    # Source : FAO/GIEWS ASIS (2017)
    modis_ref = modis[
        (modis["annee"] >= annee_ref_debut) &
        (modis["annee"] <= annee_ref_fin)
    ]

    normales_ndvi = modis_ref.groupby(
        ["departement", "mois"]
    )["ndvi_modis"].agg(
        ndvi_moy_hist = "mean",
        ndvi_std_hist = "std",
        ndvi_min_hist = "min",
        ndvi_max_hist = "max",
        n_obs         = "count"
    ).reset_index()

    # Vérification écart-type non nul
    # Si std = 0 → département sans variabilité
    # → on met une valeur minimale
    normales_ndvi["ndvi_std_hist"] = normales_ndvi[
        "ndvi_std_hist"
    ].clip(lower=0.001)

    # ── Anomalie NDVI mensuelle ───────────────────────────────
    # Pour toutes les années 2000-2024
    anomalie_ndvi = modis.merge(
        normales_ndvi[[
            "departement", "mois",
            "ndvi_moy_hist", "ndvi_std_hist"
        ]],
        on  = ["departement", "mois"],
        how = "left"
    )

    # Score Z = (valeur - moyenne) / écart-type
    # Source : FAO/GIEWS ASIS (2017)
    anomalie_ndvi["anomalie_ndvi"] = (
        (anomalie_ndvi["ndvi_modis"] -
         anomalie_ndvi["ndvi_moy_hist"])
        / anomalie_ndvi["ndvi_std_hist"]
    )

    # ── Anomalie NDVI saisonnière ─────────────────────────────
    # Moyenne de juin à octobre
    # On accorde plus de poids à septembre
    # car c'est le mois le plus prédictif
    # Source : corrélations Notebook 05

    # Poids mensuels basés sur corrélations
    # Spearman de chaque mois avec rendements
    # Notebook 05 : ndvi_sept rs=0.707
    #               ndvi_moy  rs=0.692
    # Les autres mois n'ont pas été analysés
    # séparément → poids égaux sauf septembre

    poids_mois = {
        6  : 0.15,  # Juin — début saison
        7  : 0.17,  # Juillet — croissance
        8  : 0.20,  # Août — pic pluies
        9  : 0.30,  # Septembre — pic NDVI
                    # Source : Notebook 05
        10 : 0.18   # Octobre — fin saison
    }

    anomalie_ndvi["poids_mois"] = (
        anomalie_ndvi["mois"].map(poids_mois)
    )

    # Anomalie saisonnière pondérée
    anomalie_ndvi["anomalie_ponderee"] = (
        anomalie_ndvi["anomalie_ndvi"] *
        anomalie_ndvi["poids_mois"]
    )

    anomalie_saison = anomalie_ndvi.groupby(
        ["departement", "annee"]
    ).agg(
        anomalie_ndvi_saison = (
            "anomalie_ponderee", "sum"
        ),
        ndvi_sept = (
            "ndvi_modis",
            lambda x: x.iloc[3]
            if len(x) >= 4 else np.nan
        ),
        ndvi_moy_saison = ("ndvi_modis", "mean")
    ).reset_index()

    return anomalie_ndvi, anomalie_saison, normales_ndvi


# ── Calcul anomalie NDVI ──────────────────────────────────────
anomalie_ndvi_men, anomalie_ndvi_sai, normales_ndvi = \
    calculer_anomalie_ndvi(
        modis           = modis_ref,
        annee_ref_debut = ANNEE_REF_DEBUT,
        annee_ref_fin   = ANNEE_REF_FIN
    )

# ── Résultats ─────────────────────────────────────────────────
print("=" * 65)
print("CALCUL ANOMALIE NDVI — RESULTATS")
print("Source : FAO/GIEWS ASIS (2017)")
print("         Tucker & Sellers (1986). IJRS 7(11)")
print("         Fensholt & Proud (2012). RSE 119")
print("=" * 65)

print(f"\n1. NORMALES HISTORIQUES NDVI (2000-2020)")
print(f"   Départements  : "
      f"{normales_ndvi['departement'].nunique()}")
print(f"   Observations  : {len(normales_ndvi):,}")
print(f"\n   NDVI moyen historique par mois :")
noms_mois = {
    6:"Juin", 7:"Juillet", 8:"Août",
    9:"Septembre", 10:"Octobre"
}
for mois, nom in noms_mois.items():
    moy = normales_ndvi[
        normales_ndvi["mois"] == mois
    ]["ndvi_moy_hist"].mean()
    std = normales_ndvi[
        normales_ndvi["mois"] == mois
    ]["ndvi_std_hist"].mean()
    print(f"   {nom:12} : {moy:.4f} ± {std:.4f}")

print(f"\n2. ANOMALIE NDVI MENSUELLE")
print(f"   Lignes        : {len(anomalie_ndvi_men):,}")
print(f"   Moyenne       : "
      f"{anomalie_ndvi_men['anomalie_ndvi'].mean():.3f}")
print(f"   Ecart-type    : "
      f"{anomalie_ndvi_men['anomalie_ndvi'].std():.3f}")
print(f"   Min           : "
      f"{anomalie_ndvi_men['anomalie_ndvi'].min():.3f}")
print(f"   Max           : "
      f"{anomalie_ndvi_men['anomalie_ndvi'].max():.3f}")

print(f"\n3. ANOMALIE NDVI SAISONNIERE")
print(f"   Lignes        : {len(anomalie_ndvi_sai):,}")
print(f"   Moyenne       : "
      f"{anomalie_ndvi_sai['anomalie_ndvi_saison'].mean():.3f}")
print(f"   Ecart-type    : "
      f"{anomalie_ndvi_sai['anomalie_ndvi_saison'].std():.3f}")

print(f"\n4. CLASSIFICATION ANOMALIE NDVI SAISONNIERE")
print(f"   Source : FAO/GIEWS ASIS (2017)")
classifications = [
    ("Vegetation tres degradee (< -2σ)",
     (anomalie_ndvi_sai["anomalie_ndvi_saison"] < -2).sum()),
    ("Vegetation degradee     (-2σ à -1σ)",
     ((anomalie_ndvi_sai["anomalie_ndvi_saison"] >= -2) &
      (anomalie_ndvi_sai["anomalie_ndvi_saison"] < -1)).sum()),
    ("Normale                 (-1σ à +1σ)",
     ((anomalie_ndvi_sai["anomalie_ndvi_saison"] >= -1) &
      (anomalie_ndvi_sai["anomalie_ndvi_saison"] < 1)).sum()),
    ("Vegetation bonne        (+1σ à +2σ)",
     ((anomalie_ndvi_sai["anomalie_ndvi_saison"] >= 1) &
      (anomalie_ndvi_sai["anomalie_ndvi_saison"] < 2)).sum()),
    ("Vegetation excellente   (> +2σ)",
     (anomalie_ndvi_sai["anomalie_ndvi_saison"] > 2).sum()),
]
for label, nb in classifications:
    pct = nb / len(anomalie_ndvi_sai) * 100
    print(f"   {label} : {nb:>4} ({pct:.1f}%)")

print(f"\n5. ANNEES EXTREMES NATIONALES")
ndvi_national = anomalie_ndvi_sai.groupby("annee")[
    "anomalie_ndvi_saison"
].mean()
print("   Annees les plus deficitaires :")
for annee, val in ndvi_national.sort_values().head(5).items():
    print(f"   {annee} : anomalie={val:+.3f}σ")
print("   Annees les meilleures :")
for annee, val in ndvi_national.sort_values().tail(5).items():
    print(f"   {annee} : anomalie={val:+.3f}σ")

print(f"\n6. VALIDATION — Correlation avec rendements")
print(f"   Source : Spearman (1904)")
from scipy.stats import spearmanr
dataset_valid = dataset_ml.merge(
    anomalie_ndvi_sai[[
        "departement", "annee",
        "anomalie_ndvi_saison"
    ]],
    on  = ["departement", "annee"],
    how = "left"
)
for culture in ["Arachide", "Mil"]:
    df_c = dataset_valid[
        dataset_valid["culture"] == culture
    ].dropna(subset=["anomalie_ndvi_saison"])
    rs, p = spearmanr(
        df_c["anomalie_ndvi_saison"],
        df_c["rendement"]
    )
    print(
        f"   {culture:10} : "
        f"rs={rs:.3f} | p={p:.4f} | "
        f"Sig={'OUI' if p<0.05 else 'NON'}"
    )

# ── Sauvegarde ────────────────────────────────────────────────
output = Path(
    "C:/AGRI-WATCH/data/processed/"
    "ndvi_anomalie_2000_2024.csv"
)
anomalie_ndvi_sai.to_csv(
    output, index=False, encoding="utf-8-sig"
)
logger.info(
    f"Anomalie NDVI calculee : "
    f"{len(anomalie_ndvi_sai)} lignes"
)
print(f"\nFichier sauvegarde : {output}")
print("=" * 65)

[2026-05-09 02:22:02] [INFO] [agriwatch.calcul_indicateurs] Calcul anomalie NDVI...
CALCUL ANOMALIE NDVI — RESULTATS
Source : FAO/GIEWS ASIS (2017)
         Tucker & Sellers (1986). IJRS 7(11)
         Fensholt & Proud (2012). RSE 119

1. NORMALES HISTORIQUES NDVI (2000-2020)
   Départements  : 45
   Observations  : 225

   NDVI moyen historique par mois :
   Juin         : 0.2671 ± 0.0290
   Juillet      : 0.3924 ± 0.0605
   Août         : 0.5293 ± 0.0420
   Septembre    : 0.5690 ± 0.0328
   Octobre      : 0.5220 ± 0.0316

2. ANOMALIE NDVI MENSUELLE
   Lignes        : 5,615
   Moyenne       : 0.093
   Ecart-type    : 1.015
   Min           : -3.277
   Max           : 6.409

3. ANOMALIE NDVI SAISONNIERE
   Lignes        : 1,125
   Moyenne       : 0.096
   Ecart-type    : 0.695

4. CLASSIFICATION ANOMALIE NDVI SAISONNIERE
   Source : FAO/GIEWS ASIS (2017)
   Vegetation tres degradee (< -2σ) :    2 (0.2%)
   Vegetation degradee     (-2σ à -1σ) :   70 (6.2%)
   Normale                 (-1

In [13]:
print("CLASSIFICATION ANOMALIE NDVI SAISONNIERE :")
classifications = [
    ("Vegetation tres degradee (< -2σ)",
     (anomalie_ndvi_sai["anomalie_ndvi_saison"] < -2).sum()),
    ("Vegetation degradee     (-2σ à -1σ)",
     ((anomalie_ndvi_sai["anomalie_ndvi_saison"] >= -2) &
      (anomalie_ndvi_sai["anomalie_ndvi_saison"] < -1)).sum()),
    ("Normale                 (-1σ à +1σ)",
     ((anomalie_ndvi_sai["anomalie_ndvi_saison"] >= -1) &
      (anomalie_ndvi_sai["anomalie_ndvi_saison"] < 1)).sum()),
    ("Vegetation bonne        (+1σ à +2σ)",
     ((anomalie_ndvi_sai["anomalie_ndvi_saison"] >= 1) &
      (anomalie_ndvi_sai["anomalie_ndvi_saison"] < 2)).sum()),
    ("Vegetation excellente   (> +2σ)",
     (anomalie_ndvi_sai["anomalie_ndvi_saison"] > 2).sum()),
]
for label, nb in classifications:
    pct = nb / len(anomalie_ndvi_sai) * 100
    print(f"   {label} : {nb:>4} ({pct:.1f}%)")

print("\nANNEES EXTREMES :")
ndvi_national = anomalie_ndvi_sai.groupby(
    "annee"
)["anomalie_ndvi_saison"].mean()
for annee, val in ndvi_national.sort_values().items():
    flag = ""
    if annee == 2002:
        flag = "<-- PIRE ANNEE RENDEMENTS ?"
    if annee == 2020:
        flag = "<-- MEILLEURE ANNEE RENDEMENTS ?"
    print(f"   {annee} : {val:+.3f}σ {flag}")

print("\nVALIDATION CORRELATION :")
for culture in ["Arachide", "Mil"]:
    df_c = dataset_valid[
        dataset_valid["culture"] == culture
    ].dropna(subset=["anomalie_ndvi_saison"])
    rs, p = spearmanr(
        df_c["anomalie_ndvi_saison"],
        df_c["rendement"]
    )
    print(
        f"   {culture:10} : "
        f"rs={rs:.3f} | "
        f"Sig={'OUI' if p<0.05 else 'NON'}"
    )

CLASSIFICATION ANOMALIE NDVI SAISONNIERE :
   Vegetation tres degradee (< -2σ) :    2 (0.2%)
   Vegetation degradee     (-2σ à -1σ) :   70 (6.2%)
   Normale                 (-1σ à +1σ) :  949 (84.4%)
   Vegetation bonne        (+1σ à +2σ) :  100 (8.9%)
   Vegetation excellente   (> +2σ) :    4 (0.4%)

ANNEES EXTREMES :
   2002 : -1.047σ <-- PIRE ANNEE RENDEMENTS ?
   2014 : -0.857σ 
   2018 : -0.621σ 
   2003 : -0.371σ 
   2019 : -0.274σ 
   2007 : -0.272σ 
   2017 : -0.221σ 
   2016 : -0.065σ 
   2006 : -0.012σ 
   2015 : +0.014σ 
   2004 : +0.072σ 
   2011 : +0.098σ 
   2021 : +0.142σ 
   2005 : +0.186σ 
   2013 : +0.193σ 
   2001 : +0.248σ 
   2008 : +0.309σ 
   2009 : +0.320σ 
   2000 : +0.430σ 
   2012 : +0.527σ 
   2024 : +0.590σ 
   2020 : +0.658σ <-- MEILLEURE ANNEE RENDEMENTS ?
   2010 : +0.686σ 
   2023 : +0.731σ 
   2022 : +0.930σ 

VALIDATION CORRELATION :
   Arachide   : rs=0.229 | Sig=OUI
   Mil        : rs=0.139 | Sig=OUI


In [14]:
# ============================================
# Correction anomalie NDVI
# On utilise uniquement septembre
# car c'est le mois le plus prédictif
# rs=0.707 vs rs=0.229 moyenne saisonnière
# Source : Notebook 05 corrélations Spearman
# ============================================

print("=" * 65)
print("CORRECTION ANOMALIE NDVI")
print("Utilisation NDVI septembre uniquement")
print("Justification : rs=0.707 vs rs=0.229")
print("Source : Spearman (1904) — Notebook 05")
print("=" * 65)

# ── Normales NDVI septembre uniquement ───────────────────────
modis_sept = modis_ref[modis_ref["mois"] == 9].copy()

normales_ndvi_sept = modis_sept[
    (modis_sept["annee"] >= ANNEE_REF_DEBUT) &
    (modis_sept["annee"] <= ANNEE_REF_FIN)
].groupby("departement")["ndvi_modis"].agg(
    ndvi_sept_moy = "mean",
    ndvi_sept_std = "std",
    n_obs         = "count"
).reset_index()

# Vérification écart-type non nul
normales_ndvi_sept["ndvi_sept_std"] = (
    normales_ndvi_sept["ndvi_sept_std"].clip(lower=0.001)
)

print(f"\nNormales NDVI septembre 2000-2020 :")
print(f"   Départements  : "
      f"{normales_ndvi_sept['departement'].nunique()}")
print(f"   NDVI moy sept : "
      f"{normales_ndvi_sept['ndvi_sept_moy'].mean():.4f}")
print(f"   NDVI std sept : "
      f"{normales_ndvi_sept['ndvi_sept_std'].mean():.4f}")

# ── Anomalie NDVI septembre pour toutes les années ───────────
anomalie_ndvi_sept = modis_sept.merge(
    normales_ndvi_sept[[
        "departement",
        "ndvi_sept_moy",
        "ndvi_sept_std"
    ]],
    on  = "departement",
    how = "left"
)

# Score Z = (NDVI_sept_actuel - moyenne) / écart-type
# Source : FAO/GIEWS ASIS (2017)
anomalie_ndvi_sept["anomalie_ndvi_sept"] = (
    (anomalie_ndvi_sept["ndvi_modis"] -
     anomalie_ndvi_sept["ndvi_sept_moy"])
    / anomalie_ndvi_sept["ndvi_sept_std"]
)

# ── Validation ────────────────────────────────────────────────
print(f"\nANOMALIE NDVI SEPTEMBRE :")
print(f"   Lignes     : {len(anomalie_ndvi_sept):,}")
print(f"   Moyenne    : "
      f"{anomalie_ndvi_sept['anomalie_ndvi_sept'].mean():.3f}")
print(f"   Ecart-type : "
      f"{anomalie_ndvi_sept['anomalie_ndvi_sept'].std():.3f}")

# Années extrêmes
ndvi_sept_national = anomalie_ndvi_sept.groupby("annee")[
    "anomalie_ndvi_sept"
].mean()

print(f"\nANNEES EXTREMES NDVI SEPTEMBRE :")
for annee, val in ndvi_sept_national.sort_values().items():
    flag = ""
    if annee == 2002:
        flag = "<-- PIRE ANNEE RENDEMENTS ?"
    if annee == 2020:
        flag = "<-- MEILLEURE ANNEE RENDEMENTS ?"
    print(f"   {annee} : {val:+.3f}σ {flag}")

# ── Corrélation avec rendements ───────────────────────────────
print(f"\nVALIDATION — Corrélation avec rendements")
print(f"Source : Spearman (1904)")
dataset_valid2 = dataset_ml.merge(
    anomalie_ndvi_sept[[
        "departement", "annee",
        "anomalie_ndvi_sept"
    ]],
    on  = ["departement", "annee"],
    how = "left"
)

for culture in ["Arachide", "Mil"]:
    df_c = dataset_valid2[
        dataset_valid2["culture"] == culture
    ].dropna(subset=["anomalie_ndvi_sept"])
    rs, p = spearmanr(
        df_c["anomalie_ndvi_sept"],
        df_c["rendement"]
    )
    print(
        f"   {culture:10} : "
        f"rs={rs:.3f} | "
        f"p={p:.4f} | "
        f"Sig={'OUI' if p<0.05 else 'NON'}"
    )

# ── Sauvegarde ────────────────────────────────────────────────
# On met à jour le fichier avec l'anomalie septembre
anomalie_ndvi_sai = anomalie_ndvi_sai.merge(
    anomalie_ndvi_sept[[
        "departement", "annee",
        "anomalie_ndvi_sept"
    ]],
    on  = ["departement", "annee"],
    how = "left"
)

output = Path(
    "C:/AGRI-WATCH/data/processed/"
    "ndvi_anomalie_2000_2024.csv"
)
anomalie_ndvi_sai.to_csv(
    output, index=False, encoding="utf-8-sig"
)

print(f"\nFichier mis a jour : {output}")
print("=" * 65)

logger.info(
    "Anomalie NDVI septembre calculee "
    f": {len(anomalie_ndvi_sept)} lignes"
)

CORRECTION ANOMALIE NDVI
Utilisation NDVI septembre uniquement
Justification : rs=0.707 vs rs=0.229
Source : Spearman (1904) — Notebook 05

Normales NDVI septembre 2000-2020 :
   Départements  : 45
   NDVI moy sept : 0.5690
   NDVI std sept : 0.0328

ANOMALIE NDVI SEPTEMBRE :
   Lignes     : 1,115
   Moyenne    : 0.133
   Ecart-type : 1.045

ANNEES EXTREMES NDVI SEPTEMBRE :
   2002 : -1.529σ <-- PIRE ANNEE RENDEMENTS ?
   2014 : -1.081σ 
   2018 : -0.698σ 
   2003 : -0.596σ 
   2013 : -0.472σ 
   2017 : -0.416σ 
   2005 : -0.329σ 
   2011 : -0.101σ 
   2021 : -0.046σ 
   2006 : -0.025σ 
   2019 : +0.037σ 
   2007 : +0.061σ 
   2009 : +0.081σ 
   2008 : +0.198σ 
   2016 : +0.323σ 
   2001 : +0.327σ 
   2004 : +0.334σ 
   2015 : +0.371σ 
   2000 : +0.400σ 
   2012 : +0.519σ 
   2022 : +0.930σ 
   2010 : +1.117σ 
   2024 : +1.186σ 
   2023 : +1.218σ 
   2020 : +1.371σ <-- MEILLEURE ANNEE RENDEMENTS ?

VALIDATION — Corrélation avec rendements
Source : Spearman (1904)
   Arachide   : rs=0.2

In [15]:
print("ANNEES EXTREMES COMPLETES :")
for annee, val in ndvi_sept_national.sort_values().items():
    flag = ""
    if annee == 2002:
        flag = "<-- PIRE ANNEE RENDEMENTS"
    if annee == 2020:
        flag = "<-- MEILLEURE ANNEE RENDEMENTS"
    print(f"   {annee} : {val:+.3f}σ {flag}")

print("\nCORRELATION NDVI SEPTEMBRE vs RENDEMENTS :")
for culture in ["Arachide", "Mil"]:
    df_c = dataset_valid2[
        dataset_valid2["culture"] == culture
    ].dropna(subset=["anomalie_ndvi_sept"])
    rs, p = spearmanr(
        df_c["anomalie_ndvi_sept"],
        df_c["rendement"]
    )
    print(
        f"   {culture:10} : "
        f"rs={rs:.3f} | "
        f"Sig={'OUI' if p<0.05 else 'NON'}"
    )

ANNEES EXTREMES COMPLETES :
   2002 : -1.529σ <-- PIRE ANNEE RENDEMENTS
   2014 : -1.081σ 
   2018 : -0.698σ 
   2003 : -0.596σ 
   2013 : -0.472σ 
   2017 : -0.416σ 
   2005 : -0.329σ 
   2011 : -0.101σ 
   2021 : -0.046σ 
   2006 : -0.025σ 
   2019 : +0.037σ 
   2007 : +0.061σ 
   2009 : +0.081σ 
   2008 : +0.198σ 
   2016 : +0.323σ 
   2001 : +0.327σ 
   2004 : +0.334σ 
   2015 : +0.371σ 
   2000 : +0.400σ 
   2012 : +0.519σ 
   2022 : +0.930σ 
   2010 : +1.117σ 
   2024 : +1.186σ 
   2023 : +1.218σ 
   2020 : +1.371σ <-- MEILLEURE ANNEE RENDEMENTS

CORRELATION NDVI SEPTEMBRE vs RENDEMENTS :
   Arachide   : rs=0.260 | Sig=OUI
   Mil        : rs=0.129 | Sig=OUI


In [16]:
# ============================================
# Calcul Anomalie Pluviométrique
# Normalisée
# ============================================
# L'anomalie pluviométrique normalisée mesure
# l'écart entre les précipitations actuelles
# et la normale historique du département,
# divisé par l'écart-type historique.
#
# C'est la même méthode que FAO/GIEWS ASIS
# pour la surveillance des précipitations.
#
# Méthode :
#   anomalie_pluie =
#   (pluie_totale_actuelle - moy_hist)
#   / std_hist
#
# Sources :
#   FAO/GIEWS Agricultural Stress Index
#   System (ASIS) (2017).
#   URL : fao.org/giews/earthobservation
#
#   Wilks (2011). Statistical Methods in
#   the Atmospheric Sciences. 3rd ed.
#   Academic Press.
#
#   Von Storch & Zwiers (1999). Statistical
#   Analysis in Climate Research.
#   Cambridge University Press.
# ============================================

def calculer_anomalie_pluie(
    chirps: pd.DataFrame,
    annee_ref_debut: int,
    annee_ref_fin: int
) -> tuple:
    """
    Calcule l'anomalie pluviométrique
    normalisée pour chaque département
    et chaque année.

    L'anomalie est calculée comme le score Z
    des précipitations totales saisonnières
    par rapport à la normale historique
    du département sur 2000-2020.

    Étapes :
    1. Calcul précipitations totales
       saison juin-octobre par département
    2. Calcul normale historique 2000-2020
       (moyenne + écart-type)
    3. Calcul anomalie normalisée = score Z
    4. Validation statistique

    Sources :
        FAO/GIEWS ASIS (2017).
        Wilks (2011). Academic Press.
        Von Storch & Zwiers (1999). Cambridge.

    Paramètres :
        chirps          (DataFrame) : CHIRPS
        annee_ref_debut (int)       : Début référence
        annee_ref_fin   (int)       : Fin référence

    Retourne :
        tuple : (anomalie par année,
                 normales historiques)
    """
    logger.info("Calcul anomalie pluviometrique...")

    # ── Précipitations totales saisonnières ───────────────────
    # Somme juin à octobre = saison agricole complète
    pluie_saison = chirps.groupby(
        ["departement", "annee"]
    )["precipitation_mm"].sum().reset_index()
    pluie_saison.columns = [
        "departement", "annee", "pluie_totale"
    ]

    # ── Normales historiques 2000-2020 ────────────────────────
    # Source : FAO/GIEWS ASIS (2017)
    pluie_ref = pluie_saison[
        (pluie_saison["annee"] >= annee_ref_debut) &
        (pluie_saison["annee"] <= annee_ref_fin)
    ]

    normales_pluie = pluie_ref.groupby(
        "departement"
    )["pluie_totale"].agg(
        pluie_moy_hist = "mean",
        pluie_std_hist = "std",
        pluie_min_hist = "min",
        pluie_max_hist = "max",
        n_obs          = "count"
    ).reset_index()

    # Vérification écart-type non nul
    normales_pluie["pluie_std_hist"] = (
        normales_pluie["pluie_std_hist"]
        .clip(lower=0.001)
    )

    # ── Anomalie normalisée ───────────────────────────────────
    # Score Z = (valeur - moyenne) / écart-type
    # Source : Wilks (2011)
    anomalie_pluie = pluie_saison.merge(
        normales_pluie[[
            "departement",
            "pluie_moy_hist",
            "pluie_std_hist"
        ]],
        on  = "departement",
        how = "left"
    )

    anomalie_pluie["anomalie_pluie"] = (
        (anomalie_pluie["pluie_totale"] -
         anomalie_pluie["pluie_moy_hist"])
        / anomalie_pluie["pluie_std_hist"]
    )

    return anomalie_pluie, normales_pluie


# ── Calcul ────────────────────────────────────────────────────
anomalie_pluie, normales_pluie = calculer_anomalie_pluie(
    chirps          = chirps_ref,
    annee_ref_debut = ANNEE_REF_DEBUT,
    annee_ref_fin   = ANNEE_REF_FIN
)

# ── Résultats ─────────────────────────────────────────────────
print("=" * 65)
print("CALCUL ANOMALIE PLUVIOMETRIQUE — RESULTATS")
print("Source : FAO/GIEWS ASIS (2017)")
print("         Wilks (2011). Academic Press")
print("         Von Storch & Zwiers (1999). Cambridge")
print("=" * 65)

print(f"\n1. NORMALES HISTORIQUES (2000-2020)")
print(f"   Départements       : "
      f"{normales_pluie['departement'].nunique()}")
print(f"   Pluie moy nationale: "
      f"{normales_pluie['pluie_moy_hist'].mean():.1f} mm")
print(f"   Pluie std nationale: "
      f"{normales_pluie['pluie_std_hist'].mean():.1f} mm")
print(f"\n   Top 5 départements les plus pluvieux :")
print(
    normales_pluie.nlargest(5, "pluie_moy_hist")[
        ["departement", "pluie_moy_hist", "pluie_std_hist"]
    ].to_string(index=False)
)
print(f"\n   Top 5 départements les plus secs :")
print(
    normales_pluie.nsmallest(5, "pluie_moy_hist")[
        ["departement", "pluie_moy_hist", "pluie_std_hist"]
    ].to_string(index=False)
)

print(f"\n2. ANOMALIE PLUVIOMETRIQUE")
print(f"   Lignes     : {len(anomalie_pluie):,}")
print(f"   Moyenne    : "
      f"{anomalie_pluie['anomalie_pluie'].mean():.3f}")
print(f"   Ecart-type : "
      f"{anomalie_pluie['anomalie_pluie'].std():.3f}")
print(f"   Min        : "
      f"{anomalie_pluie['anomalie_pluie'].min():.3f}")
print(f"   Max        : "
      f"{anomalie_pluie['anomalie_pluie'].max():.3f}")

print(f"\n3. ANNEES EXTREMES NATIONALES")
pluie_national = anomalie_pluie.groupby("annee")[
    "anomalie_pluie"
].mean()
for annee, val in pluie_national.sort_values().items():
    flag = ""
    if annee == 2014:
        flag = "<-- PLUS SECHE (Notebook 03)"
    if annee == 2020:
        flag = "<-- PLUS HUMIDE (Notebook 03)"
    if annee == 2002:
        flag = "<-- PIRE RENDEMENTS (Notebook 04)"
    print(f"   {annee} : {val:+.3f}σ {flag}")

print(f"\n4. CLASSIFICATION ANOMALIE PLUIE")
print(f"   Source : Wilks (2011)")
classifications = [
    ("Deficit extreme   (< -2σ)",
     (anomalie_pluie["anomalie_pluie"] < -2).sum()),
    ("Deficit severe    (-2σ à -1σ)",
     ((anomalie_pluie["anomalie_pluie"] >= -2) &
      (anomalie_pluie["anomalie_pluie"] < -1)).sum()),
    ("Normal            (-1σ à +1σ)",
     ((anomalie_pluie["anomalie_pluie"] >= -1) &
      (anomalie_pluie["anomalie_pluie"] < 1)).sum()),
    ("Excedent modere   (+1σ à +2σ)",
     ((anomalie_pluie["anomalie_pluie"] >= 1) &
      (anomalie_pluie["anomalie_pluie"] < 2)).sum()),
    ("Excedent extreme  (> +2σ)",
     (anomalie_pluie["anomalie_pluie"] > 2).sum()),
]
for label, nb in classifications:
    pct = nb / len(anomalie_pluie) * 100
    print(f"   {label} : {nb:>4} ({pct:.1f}%)")

print(f"\n5. VALIDATION — Corrélation avec rendements")
print(f"   Source : Spearman (1904)")
dataset_valid3 = dataset_ml.merge(
    anomalie_pluie[[
        "departement", "annee", "anomalie_pluie"
    ]],
    on  = ["departement", "annee"],
    how = "left"
)
for culture in ["Arachide", "Mil"]:
    df_c = dataset_valid3[
        dataset_valid3["culture"] == culture
    ].dropna(subset=["anomalie_pluie"])
    rs, p = spearmanr(
        df_c["anomalie_pluie"],
        df_c["rendement"]
    )
    print(
        f"   {culture:10} : "
        f"rs={rs:.3f} | "
        f"p={p:.4f} | "
        f"Sig={'OUI' if p<0.05 else 'NON'}"
    )

# ── Sauvegarde ────────────────────────────────────────────────
output = Path(
    "C:/AGRI-WATCH/data/processed/"
    "pluie_anomalie_2000_2024.csv"
)
anomalie_pluie.to_csv(
    output, index=False, encoding="utf-8-sig"
)
logger.info(
    f"Anomalie pluie calculee : "
    f"{len(anomalie_pluie)} lignes"
)
print(f"\nFichier sauvegarde : {output}")
print("=" * 65)

[2026-05-09 03:29:25] [INFO] [agriwatch.calcul_indicateurs] Calcul anomalie pluviometrique...
CALCUL ANOMALIE PLUVIOMETRIQUE — RESULTATS
Source : FAO/GIEWS ASIS (2017)
         Wilks (2011). Academic Press
         Von Storch & Zwiers (1999). Cambridge

1. NORMALES HISTORIQUES (2000-2020)
   Départements       : 45
   Pluie moy nationale: 654.6 mm
   Pluie std nationale: 106.6 mm

   Top 5 départements les plus pluvieux :
departement  pluie_moy_hist  pluie_std_hist
   Oussouye     1347.738095      251.358795
 Ziguinchor     1321.118571      226.591475
    Bignona     1177.555714      208.379390
   Salémata     1155.205238      184.898882
    Goudomp     1146.015238      153.589687

   Top 5 départements les plus secs :
departement  pluie_moy_hist  pluie_std_hist
     Dagana      253.761905       59.290059
Saint-Louis      290.659048       74.704863
      Podor      293.881429       64.009469
      Louga      330.013810       72.572400
      Matam      345.179524       73.300522

2. ANO

In [17]:
print("ANNEES EXTREMES COMPLETES :")
for annee, val in pluie_national.sort_values().items():
    flag = ""
    if annee == 2014:
        flag = "<-- PLUS SECHE (Notebook 03)"
    if annee == 2020:
        flag = "<-- PLUS HUMIDE (Notebook 03)"
    if annee == 2002:
        flag = "<-- PIRE RENDEMENTS (Notebook 04)"
    print(f"   {annee} : {val:+.3f}σ {flag}")

print("\nCLASSIFICATION ANOMALIE PLUIE :")
classifications = [
    ("Deficit extreme   (< -2σ)",
     (anomalie_pluie["anomalie_pluie"] < -2).sum()),
    ("Deficit severe    (-2σ à -1σ)",
     ((anomalie_pluie["anomalie_pluie"] >= -2) &
      (anomalie_pluie["anomalie_pluie"] < -1)).sum()),
    ("Normal            (-1σ à +1σ)",
     ((anomalie_pluie["anomalie_pluie"] >= -1) &
      (anomalie_pluie["anomalie_pluie"] < 1)).sum()),
    ("Excedent modere   (+1σ à +2σ)",
     ((anomalie_pluie["anomalie_pluie"] >= 1) &
      (anomalie_pluie["anomalie_pluie"] < 2)).sum()),
    ("Excedent extreme  (> +2σ)",
     (anomalie_pluie["anomalie_pluie"] > 2).sum()),
]
for label, nb in classifications:
    pct = nb / len(anomalie_pluie) * 100
    print(f"   {label} : {nb:>4} ({pct:.1f}%)")

print("\nVALIDATION CORRELATION :")
for culture in ["Arachide", "Mil"]:
    df_c = dataset_valid3[
        dataset_valid3["culture"] == culture
    ].dropna(subset=["anomalie_pluie"])
    rs, p = spearmanr(
        df_c["anomalie_pluie"],
        df_c["rendement"]
    )
    print(
        f"   {culture:10} : "
        f"rs={rs:.3f} | "
        f"Sig={'OUI' if p<0.05 else 'NON'}"
    )

ANNEES EXTREMES COMPLETES :
   2014 : -1.451σ <-- PLUS SECHE (Notebook 03)
   2011 : -0.804σ 
   2019 : -0.790σ 
   2007 : -0.772σ 
   2018 : -0.648σ 
   2004 : -0.577σ 
   2001 : -0.465σ 
   2002 : -0.461σ <-- PIRE RENDEMENTS (Notebook 04)
   2006 : -0.406σ 
   2017 : -0.190σ 
   2016 : -0.170σ 
   2013 : -0.117σ 
   2000 : +0.063σ 
   2003 : +0.071σ 
   2021 : +0.306σ 
   2008 : +0.475σ 
   2024 : +0.477σ 
   2023 : +0.486σ 
   2009 : +0.760σ 
   2005 : +0.830σ 
   2015 : +0.936σ 
   2012 : +1.015σ 
   2022 : +1.196σ 
   2010 : +1.247σ 
   2020 : +1.455σ <-- PLUS HUMIDE (Notebook 03)

CLASSIFICATION ANOMALIE PLUIE :
   Deficit extreme   (< -2σ) :    9 (0.8%)
   Deficit severe    (-2σ à -1σ) :  140 (12.4%)
   Normal            (-1σ à +1σ) :  769 (68.4%)
   Excedent modere   (+1σ à +2σ) :  173 (15.4%)
   Excedent extreme  (> +2σ) :   34 (3.0%)

VALIDATION CORRELATION :
   Arachide   : rs=0.229 | Sig=OUI
   Mil        : rs=0.171 | Sig=OUI


In [18]:
print("TEST CRUCIAL — Anomalies vs Anomalies")

TEST CRUCIAL — Anomalies vs Anomalies
